# Fashion MNIST MLP Task

In [1]:
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.datasets import FashionMNIST
from torchvision import transforms
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Set seed values to compare the same split/training behavior
SEED = 123
random.seed(SEED)
torch.manual_seed(SEED)

In [2]:
transform = transforms.ToTensor()
full_dataset = FashionMNIST(root='./data', train=True, download=True, transform=transform)

subset_size = 12000
subset_indices = torch.randperm(len(full_dataset), generator=torch.Generator().manual_seed(SEED))[:subset_size].tolist()
subset = torch.utils.data.Subset(full_dataset, subset_indices)

train_size = int(0.8 * subset_size)
test_size = subset_size - train_size
train_subset, test_subset = random_split(
    subset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_subset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=128, shuffle=False)

print(f'Subset size: {subset_size}')
print(f'Train size: {len(train_subset)}')
print(f'Test size: {len(test_subset)}')

100.0%
100.0%
100.0%
100.0%

Subset size: 12000
Train size: 9600
Test size: 2400


In [3]:
class MLP(nn.Module):
    def __init__(self, hidden_units=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, 10)
        )

    def forward(self, x):
        return self.net(x)


def train_model(model, train_loader, test_loader, epochs=8, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for _ in range(epochs):
        model.train()
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())

    return accuracy_score(all_labels, all_preds)

Chosen hyperparameter: hidden layer size (`hidden_units`).

Reason: it directly controls model capacity, so it is a simple way to improve or reduce classification performance for an MLP.

In [4]:
hidden_unit_values = [64, 128, 256]
results = {}
best_acc = -1
best_hidden_units = None
best_model = None

for hu in hidden_unit_values:
    model_candidate = MLP(hidden_units=hu)
    acc = train_model(model_candidate, train_loader, test_loader, epochs=8, lr=0.001)
    results[hu] = acc
    print(f'hidden_units={hu}, test_accuracy={acc:.4f}')

    if acc > best_acc:
        best_acc = acc
        best_hidden_units = hu
        best_model = model_candidate

model = best_model
print(f'\nSelected hidden_units={best_hidden_units} because it gave the highest test accuracy: {best_acc:.4f}')

if best_acc < 0.80:
    print('Accuracy is below 80%. Re-training with more epochs.')
    model = MLP(hidden_units=best_hidden_units)
    retry_acc = train_model(model, train_loader, test_loader, epochs=12, lr=0.001)
    print(f'Retry accuracy: {retry_acc:.4f}')

hidden_units=64, test_accuracy=0.8275
hidden_units=128, test_accuracy=0.8304
hidden_units=256, test_accuracy=0.8462

Selected hidden_units=256 because it gave the highest test accuracy: 0.8462


In [5]:
def evaluate_model(model, test_loader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())
    return all_preds, all_labels

preds, labels = evaluate_model(model, test_loader)
conf_matrix = confusion_matrix(labels, preds)
print('Confusion Matrix:')
print(conf_matrix)

acc = accuracy_score(labels, preds)
prec = precision_score(labels, preds, average='macro')
rec = recall_score(labels, preds, average='macro')
f1 = f1_score(labels, preds, average='macro')

print(f'\nAccuracy:  {acc:.4f}')
print(f'Precision (macro): {prec:.4f}')
print(f'Recall (macro):    {rec:.4f}')
print(f'F1-score (macro):  {f1:.4f}')

Confusion Matrix:
[[185   1   8  11   0   0  14   0   3   0]
 [  1 221   2   4   0   0   0   0   1   0]
 [  4   0 175   4  27   0  11   0   1   0]
 [  6   2   2 230  14   0   4   0   0   0]
 [  0   0  24  15 187   0   8   0   1   0]
 [  0   0   0   1   0 222   0  12   2   5]
 [ 48   2  40   8  33   0 129   0   7   0]
 [  0   0   0   0   0   6   0 244   0   4]
 [  0   0   2   3   1   1   1   0 235   0]
 [  0   0   0   0   0   4   0  21   0 203]]

Accuracy:  0.8462
Precision (macro): 0.8479
Recall (macro):    0.8492
F1-score (macro):  0.8441


In [6]:
class_names = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

cm_no_diag = conf_matrix.copy()
for i in range(cm_no_diag.shape[0]):
    cm_no_diag[i, i] = 0

worst_pairs = []
for true_class in range(cm_no_diag.shape[0]):
    pred_class = cm_no_diag[true_class].argmax()
    count = cm_no_diag[true_class, pred_class]
    worst_pairs.append((count, true_class, pred_class))

worst_pairs.sort(reverse=True)
print('Classes the model struggles with most (highest off-diagonal counts):')
for count, true_class, pred_class in worst_pairs[:5]:
    if count > 0:
        print(f"True '{class_names[true_class]}' predicted as '{class_names[pred_class]}' -> {count} times")

Classes the model struggles with most (highest off-diagonal counts):
True 'Shirt' predicted as 'T-shirt/top' -> 48 times
True 'Pullover' predicted as 'Coat' -> 27 times
True 'Coat' predicted as 'Pullover' -> 24 times
True 'Ankle boot' predicted as 'Sneaker' -> 21 times
True 'Dress' predicted as 'Coat' -> 14 times
